# Distributed Training on Snowflake

**Session 3 — Deep Dive: Distributed Training & Hyperparameter Optimization**  
**Notebook 3 of 4** | ~15 minutes

---

## What You Will Learn

| Concept | What We Cover |
|---------|---------------|
| **Multi-node ML Jobs** | Set `target_instances > 1` to distribute across nodes |
| **Distributed Estimators** | `XGBEstimator` for native distributed XGBoost |
| **DataConnector** | Efficient data loading for distributed training |
| **Scaling Config** | Control workers, CPUs, and GPUs per node |
| **When to distribute** | Decision framework for single vs multi-node |

---

## When to Use Distributed Training

| Scenario | Recommendation |
|----------|----------------|
| Dataset < 10 GB, trains in < 30 min | **Single node** — simpler, no coordination overhead |
| Dataset > 10 GB or training > 1 hour | **Multi-node** — partition data across workers |
| Deep learning (PyTorch) | **Multi-GPU** — use `PyTorchDistributor` |
| Gradient boosting at scale | **`XGBEstimator` / `LightGBMEstimator`** — native distributed |

### Snowflake's Distributed Training Options

```
 ┌─────────────────────────────────────────────────┐
 │           Snowflake Distributed Training         │
 │                                                  │
 │  ┌──────────────┐  ┌──────────────┐             │
 │  │ XGBEstimator │  │ LightGBM     │  Gradient   │
 │  │ (distributed │  │ Estimator    │  Boosting   │
 │  │  XGBoost)    │  │              │             │
 │  └──────────────┘  └──────────────┘             │
 │                                                  │
 │  ┌──────────────┐  ┌──────────────┐             │
 │  │ PyTorch      │  │ Many Model   │  DL / Multi │
 │  │ Distributor  │  │ Training     │  -Partition  │
 │  │ (DDP)        │  │ (per-segment)│             │
 │  └──────────────┘  └──────────────┘             │
 │                                                  │
 │  All powered by Ray on SPCS compute pools        │
 └─────────────────────────────────────────────────┘
```

## 1 | Environment Setup

In [ ]:
%cd ..
%load_ext autoreload

In [ ]:
import json
import logging
import os

from utils import get_config
from utils import get_session

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

config = get_config("config.yaml")
DB = config.snowflake.database
SCHEMA = config.snowflake.schema_name
WH = config.snowflake.warehouse
COMPUTE_POOL = config.compute.compute_pool
STAGE = f"{DB}.{SCHEMA}.{config.stages.job_payloads}"

session = get_session(
    connection_name=config.snowflake.connection_name,
    database_name=DB,
    schema_name=SCHEMA,
    warehouse_name=WH,
)

print(f"Connected as {session.get_current_user()} | Role: {session.get_current_role()}")
print(f"Compute Pool: {COMPUTE_POOL}")

## 2 | Multi-Node ML Jobs — Key Concepts

Setting `target_instances > 1` provisions multiple containers. Snowflake sets
`RANK` and `WORLD_SIZE` environment variables on each node.

```
 submit_directory(target_instances=3)
 │
 ├── Node 0 (RANK=0, WORLD_SIZE=3)  ← Head node
 │   └── python train.py
 │
 ├── Node 1 (RANK=1, WORLD_SIZE=3)  ← Worker
 │   └── python train.py
 │
 └── Node 2 (RANK=2, WORLD_SIZE=3)  ← Worker
     └── python train.py
```

### Important: Your Model Must Be Distribution-Aware

Simply setting `target_instances=2` does **not** automatically parallelize your model.
Each node runs the same script independently. Unless the model framework coordinates
across nodes, you just get N copies of the same training run.

| Model Type | Multi-node benefit? | How to distribute |
|---|---|---|
| sklearn (RF, LogReg, SVM) | **No** — not distributed-aware | Use single instance |
| XGBoost / LightGBM | **Yes** | Use `XGBEstimator` / `LightGBMEstimator` |
| PyTorch (deep learning) | **Yes** | Use `PyTorchDistributor` (DDP across GPUs) |
| Any model (HPO) | **Yes** | Run parallel trials via Tuner API / Ray Tune |
| Per-segment models | **Yes** | Use `ManyModelTraining` (one model per partition) |

### What If Your Model Isn't Available as a Snowflake Estimator?

If you need to distribute a model that Snowflake doesn't have a built-in estimator for,
you have two options:

1. **Ray directly** — Write a custom Ray script that initializes a Ray cluster using
   `RANK`/`WORLD_SIZE` env vars, then use Ray's distributed primitives (`ray.train`,
   `ray.data`) to shard and train. Your entrypoint checks `RANK == 0` to start the head
   node and `RANK > 0` to start workers.

2. **PyTorchDistributor** — If your model is PyTorch-based, wrap it in
   `PyTorchDistributor` which handles DDP setup and `ShardedDataConnector` for
   data partitioning automatically.

In both cases, **your script must explicitly coordinate between nodes** — Snowflake
provides the infrastructure (multiple containers, network connectivity, env vars)
but the distribution logic lives in your code or the framework you choose.

## 3 | Distributed XGBoost with `XGBEstimator`

For gradient boosting, Snowflake provides **native distributed estimators** that handle
data partitioning, worker coordination, and model aggregation automatically.

> **Documentation**: [Snowflake Distributed Training](https://docs.snowflake.com/en/developer-guide/snowflake-ml/distributed-training)

### `XGBEstimator` vs raw XGBoost

| Aspect | Raw XGBoost | XGBEstimator |
|--------|-------------|---------------|
| Data loading | Manual `to_pandas()` | Automatic via `DataConnector` |
| Distribution | Write your own Ray code | Built-in, auto-configured |
| GPU support | Manual device management | `use_gpu=True` |
| Worker scaling | Manual Ray cluster setup | `XGBScalingConfig` |

### `DataConnector` — Efficient Data Loading

`DataConnector` is the bridge between Snowflake tables and distributed training.
It handles partitioning, shuffling, and streaming data to workers efficiently.

```python
from snowflake.ml.data.data_connector import DataConnector

train_connector = DataConnector.from_dataframe(session.table('TRAINING_FEATURES'))
```

> **Important**: `XGBEstimator` and `DataConnector` run **server-side** on SPCS.

### Scaling Configuration Deep Dive

| Parameter | Default | Description |
|-----------|---------|-------------|
| `num_workers` | -1 (auto) | Number of Ray worker processes |
| `num_cpu_per_worker` | -1 (auto) | CPUs allocated per worker |
| `use_gpu` | None (auto) | Enable GPU training |

#### Auto-Detection (-1 / None)

When you use the defaults, Snowflake inspects the compute pool's instance family
and automatically configures the optimal worker count and resource allocation.

| Instance Family | Auto Workers | Auto CPUs/Worker |
|----------------|-------------|------------------|
| CPU_X64_S (4 vCPU) | 2 | 2 |
| CPU_X64_M (12 vCPU) | 4 | 3 |
| GPU_NV_S (1 GPU) | 1 | All CPUs |
| GPU_NV_M (4 GPU) | 4 | CPUs / 4 |

## 4 | Submit the Distributed XGBoost Job

Now let's run `train_distributed.py` from the `job_payload/` directory —
using `submit_directory` with `target_instances=2`.

In [ ]:
from snowflake.ml.jobs import submit_directory

JOB_DIR = "session3_distributed_training_and_hpo/job_payload"
NUM_NODES = 2

job = submit_directory(
    dir_path=JOB_DIR,
    entrypoint="train_distributed.py",
    compute_pool=COMPUTE_POOL,
    stage_name=STAGE,
    target_instances=NUM_NODES,
    env_vars={
        "DATABASE": DB,
        "SCHEMA": SCHEMA,
        "MODEL_NAME": "PATIENT_RISK_XGB_DISTRIBUTED",
        "N_ESTIMATORS": "200",
        "MAX_DEPTH": "8",
        "LEARNING_RATE": "0.1",
    },
)

print(f"Distributed XGBoost job submitted ({NUM_NODES} nodes)")
print(f"  Job ID: {job.id}")
print(f"  Status: {job.status}")

### View the Ray Dashboard

NOTE: The dashboard URL can takes a few minutes to become available.

In [ ]:
job.get_ray_dashboard_url()

### Print Job Status and Logs

In [ ]:
print(f"Waiting for job {job.id} ...")
job.wait()
print(f"Final status: {job.status}")

logs = job.get_logs()
print("\n" + "=" * 60)
print("JOB LOGS (tail)")
print("=" * 60)
print(logs[-2000:] if len(logs) > 2000 else logs)

In [ ]:
from snowflake.ml.registry import Registry

registry = Registry(session, database_name=DB, schema_name=SCHEMA)

model = registry.get_model("PATIENT_RISK_XGB_DISTRIBUTED")
print(f"Model: {model.name}")
v = model.last()
metrics = v.show_metrics()
print(f"  {v.version_name} — mlogloss: {metrics.get('mlogloss', 'N/A')}")

## 5 | Best Practices for Distributed Training

### Data Loading

| Pattern | When | How |
|---------|------|-----|
| `DataConnector.from_dataframe()` | Standard distributed estimators | Automatic partitioning |
| `ShardedDataConnector` | PyTorch DDP | Each worker gets a unique data shard |
| `to_pandas()` in script | Small data + multi-node | Each node loads full dataset |

### Resource Sizing

| Data Size | Recommended Setup |
|-----------|-------------------|
| < 1 GB | 1 node, CPU_X64_S |
| 1-10 GB | 2-3 nodes, CPU_X64_M |
| 10-100 GB | 3-6 nodes, CPU_X64_M or HIGHMEM_X64_S |
| > 100 GB | 5+ nodes, HIGHMEM_X64_M |
| Deep learning | GPU_NV_S/M, use `PyTorchDistributor` |

### Common Pitfalls

| Pitfall | Solution |
|---------|----------|
| Too many small nodes | Fewer larger nodes reduce coordination overhead |
| `to_pandas()` on huge tables | Use `DataConnector` for streaming ingestion |
| Not checking instance logs | Use `job.get_logs(instance_id=N)` to debug specific nodes |
| Forgetting `Session.builder.getOrCreate()` | Required inside SPCS — no connection params needed |

## 6 | Summary

In this notebook we:

1. **Learned** when multi-node training helps and when it doesn't
2. **Explored** `XGBEstimator` for native distributed XGBoost training
3. **Ran** distributed XGBoost via `train_distributed.py` with `target_instances=2`
4. **Verified** the model in the registry
5. **Reviewed** best practices for data loading and resource sizing

### Key Takeaways

| Takeaway | Detail |
|----------|--------|
| **Distribution-aware models** | Only models with built-in coordination benefit from multi-node |
| **`XGBEstimator`** | Native distributed XGBoost — no Ray code required |
| **`DataConnector`** | Efficient bridge from Snowflake tables to distributed workers |
| **Custom distribution** | Use Ray directly or `PyTorchDistributor` for unsupported frameworks |
| **Auto-detection** | Workers, CPUs, and GPUs detected from compute pool |

---

**Next →** [04_hyperparameter_optimization.ipynb](04_hyperparameter_optimization.ipynb) — Distributed HPO with the Tuner API